# Real pretrained LoRA SFT and teacher distillation
Runs one baseline and two adapter conditions on identical held-out tasks. The oracle condition uses labeled synthetic solutions; teacher labels come from actual model rollouts and only successful train trajectories are accepted. Zero accepted teacher trajectories is a valid reported outcome: teacher SFT is skipped, never replaced by oracle data. This notebook has **not** been run on pretrained weights in the repository evidence.


## Runtime and provenance
Select a free GPU runtime manually. This notebook never provisions paid resources. Default models fit a typical T4; memory use varies. All pretrained weights download in your Colab runtime, not in checked-in validation. Each run saves resolved Hub commits, package versions, seeds, budgets, task IDs and outputs. The code resolves mutable `main` to one commit per model before loading. Save the archive before disconnecting; Colab storage is temporary.


In [ ]:
import sys, subprocess, os
from pathlib import Path
REPO = Path('/content/toollearn')
if not REPO.exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/hiroshi-os/toollearn.git', str(REPO)])
os.chdir(REPO)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.6', 'peft==0.21.2', 'accelerate>=1,<2'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'])
print('Source commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
import torch, platform, importlib.metadata
print({'python': platform.python_version(), 'torch': torch.__version__,
       'transformers': importlib.metadata.version('transformers'),
       'peft': importlib.metadata.version('peft'), 'cuda': torch.cuda.is_available()})
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, then rerun. No provisioning is performed.'
free, total = torch.cuda.mem_get_info()
print(torch.cuda.get_device_name(0), 'free GiB:', round(free/2**30, 2), 'total GiB:', round(total/2**30, 2))
assert free >= 6*2**30, 'This modest default needs at least 6 GiB free. Restart runtime or reduce model/budgets.'
# This is a guard, not an exact memory estimate. OOM remains possible with larger settings.


## Configure a modest pilot
Keep held-out test settings fixed. Tune on dev in separate runs; this pilot is one seed and cannot establish an effect. The optional chat template must be shared across collection, training and evaluation. This notebook enables model chat templates by default; unsupported templates fail explicitly. Plain role text remains an optional comparison and may disadvantage instruction models. Teacher inference is bounded by count × max tool calls × output tokens. Successful-filtering acceptance and selection bias must accompany results. Teacher dataset reuse validates config and SHA-256; changing collection settings requires a new output directory.


In [ ]:
OUTPUT = 'artifacts/colab-adaptation'
MODEL = 'HuggingFaceTB/SmolLM2-135M-Instruct'
TEACHER = 'Qwen/Qwen2.5-0.5B-Instruct'
SEED, TRAIN_TASKS, TEST_TASKS, STEPS = 17, 16, 8, 20
CHAT_TEMPLATE = True
cmd = [sys.executable, '-m', 'toollearn.studies', '--output', OUTPUT,
       '--model', MODEL, '--teacher', TEACHER, '--seed', str(SEED),
       '--count', str(TRAIN_TASKS), '--test-count', str(TEST_TASKS),
       '--steps', str(STEPS), '--rank', '8', '--max-steps', '8', '--max-new-tokens', '160']
if CHAT_TEMPLATE: cmd.append('--chat-template')
# Model phases release GPU objects between teacher/student/evaluation conditions.
subprocess.check_call(cmd)


In [ ]:
import json
print(json.dumps(json.loads(Path(OUTPUT, 'comparison.json').read_text()), indent=2))
print('Artifacts:', sorted(p.name for p in Path(OUTPUT).iterdir()))
# oracle-adapter/ and, only when accepted > 0, teacher-adapter/ contain PEFT checkpoints.


## After the pilot: optional replication
A 20-update single-seed pilot only checks feasibility. The opt-in sweep below uses three seeds and two train-set sizes with fixed held-outs and 100 updates. This takes materially longer. Report each seed and teacher acceptance separately, aggregate uncertainty, and use dev for any tuning before test.

In [ ]:
RUN_SWEEP = False  # Enable only when ready for the larger experiment.
if RUN_SWEEP:
    for seed in [17, 29, 43]:
        for count in [32, 128]:
            sweep = cmd.copy()
            for flag, value in [('--output', f'artifacts/sweep-seed{seed}-n{count}'),
                                ('--seed', str(seed)), ('--count', str(count)), ('--steps', '100')]:
                sweep[sweep.index(flag) + 1] = value
            subprocess.check_call(sweep)
    # Download sweep outputs separately before the runtime disconnects.


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive('/content/toollearn-results', 'zip', OUTPUT)
files.download(archive)
